# LeetCode 604 — Design Compressed String Iterator bằng Python

**Link đề:** https://leetcode.com/problems/design-compressed-string-iterator/
**Độ khó:** Easy · **Chủ đề:** Array, String, Design, Iterator

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class StringIterator:

    def __init__(self, compressedString: str):
        ...

    def next(self) -> str:
        ...

    def hasNext(self) -> bool:
        ...

# Your StringIterator object will be instantiated and called as such:
# obj = StringIterator(compressedString)
# param_1 = obj.next()
# param_2 = obj.hasNext()
```

## Mục tiêu học
1. Hiểu dạng bài **Design** trên LeetCode: viết một `class` có trạng thái, LeetCode gọi lần lượt các phương thức.
2. Ôn `class`, `__init__`, `self`, thuộc tính; viết hàm "chạy thử" giống cách LeetCode chấm bài Design.
3. Tách chuỗi nén `"L1e2t1"` thành các cặp `(ký tự, số lần)` – xử lý số **nhiều chữ số** bằng `isdigit` và công thức `num = num * 10 + digit`.
4. Đi từ **giải nén toàn bộ** (hỏng khi số lần tới `10^9`) → **danh sách cặp + con trỏ** → **phân tích lười (lazy) trực tiếp trên chuỗi**.
5. Trace trạng thái sau mỗi lời gọi, chứng minh invariant, Time/Space, 6 bài tập có đáp án cuối notebook.

## 1. Đề bài và ví dụ

Thiết kế cấu trúc dữ liệu **iterator** (bộ duyệt) cho một **chuỗi nén**. Chuỗi nén có dạng: mỗi **chữ cái** đứng trước một **số nguyên dương** cho biết chữ cái đó lặp bao nhiêu lần trong chuỗi gốc.

Ví dụ `"L1e2t1C1o1d1e1"` là chuỗi gốc `"LeetCode"`.

Cài đặt class `StringIterator`:
- `StringIterator(compressedString)`: khởi tạo với chuỗi nén.
- `next()`: trả về **ký tự kế tiếp** của chuỗi gốc nếu vẫn còn; nếu hết thì trả về **một dấu cách** `" "`.
- `hasNext()`: trả `True` nếu còn ký tự chưa đọc, ngược lại `False`.

**Ví dụ:**

```text
Input:  ["StringIterator", "next", "next", "next", "next", "next", "next", "hasNext", "next", "hasNext"]
        [["L1e2t1C1o1d1e1"], [], [], [], [], [], [], [], [], []]
Output: [null, "L", "e", "e", "t", "C", "o", true, "d", true]
```

Giải thích: chuỗi gốc là `L e e t C o d e`. Sau 6 lần `next` đã đọc `L,e,e,t,C,o`; `hasNext` → `True`; `next` → `d`; `hasNext` → `True` (còn `e`).

**Constraints (theo hiểu biết về bài):**
- `1 <= compressedString.length <= 1000`.
- Chuỗi gồm chữ cái thường/hoa tiếng Anh và chữ số.
- Số lần lặp của một ký tự nằm trong `[1, 10^9]`.
- Tối đa `100` lời gọi `next` và `hasNext`.

Điểm mấu chốt từ constraints: số lần lặp có thể **tới một tỷ** → **không được giải nén ra chuỗi thật**.

## 2. Nền tảng: `class`, `__init__`, `self` và cách LeetCode chấm bài Design

### 2.1. Class có trạng thái
- `class` là "khuôn" để tạo đối tượng. `__init__` chạy **một lần** khi tạo đối tượng.
- `self` là chính đối tượng đó; `self.x = ...` lưu **trạng thái** sống qua các lần gọi phương thức.

Ví dụ mini: một bộ đếm.

In [1]:
class Counter3:
    def __init__(self, start):
        self.value = start          # trạng thái

    def step(self):
        self.value += 1
        return self.value

c = Counter3(10)
print(c.step(), c.step(), c.step())   # 11 12 13: trạng thái được nhớ giữa các lần gọi

11 12 13


### 2.2. Mô phỏng cách LeetCode chạy bài Design
LeetCode đưa hai list: tên thao tác và đối số. Thao tác đầu tiên là **tạo đối tượng** (kết quả `null` = `None`), các thao tác sau gọi phương thức. Ta viết hàm `run_ops` để chạy giống vậy.

`getattr(obj, "next")` lấy phương thức theo **tên chuỗi**; `f(*args)` gọi với các đối số trong list.

In [2]:
def run_ops(cls, ops, args):
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == cls.__name__ or op == "StringIterator":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print(list(zip(["x", "y"], [1, 2])))        # zip ghép từng cặp
print(getattr("abc", "upper")())            # gọi "abc".upper()

[('x', 1), ('y', 2)]
ABC


### 2.3. Đọc số nhiều chữ số trong chuỗi
- `ch.isdigit()` kiểm tra ký tự là chữ số; `ch.isalpha()` kiểm tra chữ cái.
- `int(ch)` đổi `"7"` → `7`.
- Đọc `"125"` từ trái sang phải: `num = num * 10 + digit` → `1 → 12 → 125`.

In [3]:
s = "a125"
print("'a'.isalpha():", s[0].isalpha(), "| '1'.isdigit():", s[1].isdigit())
num = 0
for ch in s[1:]:
    num = num * 10 + int(ch)
    print("đọc", ch, "-> num =", num)

'a'.isalpha(): True | '1'.isdigit(): True
đọc 1 -> num = 1
đọc 2 -> num = 12
đọc 5 -> num = 125


### 2.4. Tách chuỗi nén thành các cặp `(ký tự, số lần)`
Duyệt với chỉ số `i`: tại `i` là chữ cái; tăng `i`, đọc **hết** các chữ số liên tiếp để ra số lần.

In [4]:
def parse_pairs(compressed):
    pairs = []
    i, n = 0, len(compressed)
    while i < n:
        ch = compressed[i]            # chữ cái
        i += 1
        num = 0
        while i < n and compressed[i].isdigit():
            num = num * 10 + int(compressed[i])
            i += 1
        pairs.append((ch, num))
    return pairs

print(parse_pairs("L1e2t1C1o1d1e1"))
print(parse_pairs("a12B3c1000000000"))

[('L', 1), ('e', 2), ('t', 1), ('C', 1), ('o', 1), ('d', 1), ('e', 1)]
[('a', 12), ('B', 3), ('c', 1000000000)]


## 3. Cách 1 — Giải nén toàn bộ (baseline dễ nghĩ)

Dựng chuỗi gốc bằng `ch * num` (nhân chuỗi = lặp lại) rồi dùng một con trỏ `pos`.

`"ab" * 3 == "ababab"`; `"".join(list_chuoi)` nối các chuỗi lại.

In [5]:
class StringIteratorExpand:
    def __init__(self, compressedString: str):
        self.text = "".join(ch * num for ch, num in parse_pairs(compressedString))
        self.pos = 0

    def next(self) -> str:
        if not self.hasNext():
            return " "
        ch = self.text[self.pos]
        self.pos += 1
        return ch

    def hasNext(self) -> bool:
        return self.pos < len(self.text)


OPS = ["StringIterator", "next", "next", "next", "next", "next", "next", "hasNext", "next", "hasNext"]
ARGS = [["L1e2t1C1o1d1e1"], [], [], [], [], [], [], [], [], []]
EXPECTED = [None, "L", "e", "e", "t", "C", "o", True, "d", True]
print(run_ops(StringIteratorExpand, OPS, ARGS))

[None, 'L', 'e', 'e', 't', 'C', 'o', True, 'd', True]


**Nút thắt:** `"a1000000000"` sẽ tạo chuỗi **1 tỷ ký tự** (~1 GB) chỉ để đọc tối đa 100 ký tự! Ta thử với số lớn vừa phải để thấy độ dài chuỗi giải nén tăng theo số lần lặp chứ không theo độ dài chuỗi nén.

In [6]:
compressed = "a5000000"
it = StringIteratorExpand(compressed)
print("Chuỗi nén dài", len(compressed), "ký tự -> chuỗi giải nén dài", len(it.text), "ký tự")
print("Nếu là a1000000000 thì cần ~", 10**9 // 10**6, "MB chỉ để lưu chuỗi")

Chuỗi nén dài 8 ký tự -> chuỗi giải nén dài 5000000 ký tự
Nếu là a1000000000 thì cần ~ 1000 MB chỉ để lưu chuỗi


## 4. Insight: chỉ cần nhớ "đang ở cặp nào, còn lại bao nhiêu"

Không cần chuỗi gốc. Trạng thái đủ dùng gồm:
- danh sách cặp `pairs` (dài tối đa ~500 cặp vì chuỗi nén ≤ 1000 ký tự);
- `idx`: đang ở cặp thứ mấy;
- `remaining`: ký tự của cặp hiện tại còn bao nhiêu lần.

`next()`:
1. Nếu hết (`not hasNext()`) → `" "`.
2. Lấy `ch = pairs[idx][0]`, giảm `remaining`.
3. Nếu `remaining == 0` → chuyển sang cặp sau: `idx += 1`, `remaining = pairs[idx][1]` (nếu còn).

`hasNext()`: còn ký tự ⇔ `remaining > 0` (vì ta luôn chuyển cặp ngay khi hết).

## 5. Cách 2 — Danh sách cặp + con trỏ: giảng từng đoạn rồi ghép

In [7]:
class StringIteratorPairs:
    def __init__(self, compressedString: str):
        self.pairs = [(ch, k) for ch, k in parse_pairs(compressedString) if k > 0]
        self.idx = 0
        self.remaining = self.pairs[0][1] if self.pairs else 0

    def next(self) -> str:
        if not self.hasNext():
            return " "
        ch = self.pairs[self.idx][0]
        self.remaining -= 1
        if self.remaining == 0:                 # cặp hiện tại đã dùng hết
            self.idx += 1
            if self.idx < len(self.pairs):
                self.remaining = self.pairs[self.idx][1]
        return ch

    def hasNext(self) -> bool:
        return self.remaining > 0


print(run_ops(StringIteratorPairs, OPS, ARGS))
big = StringIteratorPairs("a1000000000b2")
print("Số lần lặp 10^9 vẫn tạo tức thì:", big.next(), big.next(), big.remaining)

[None, 'L', 'e', 'e', 't', 'C', 'o', True, 'd', True]
Số lần lặp 10^9 vẫn tạo tức thì: a a 999999998


### 5.1. TRACE trạng thái sau mỗi lời gọi
Chuỗi `"a2b1c3"` → `pairs = [(a,2), (b,1), (c,3)]`.

| Lời gọi | idx trước | remaining trước | trả về | idx sau | remaining sau |
|---|---:|---:|---|---:|---:|
| next | 0 | 2 | `a` | 0 | 1 |
| next | 0 | 1 | `a` | 1 | 1 |
| next | 1 | 1 | `b` | 2 | 3 |
| hasNext | 2 | 3 | `True` | 2 | 3 |
| next | 2 | 3 | `c` | 2 | 2 |
| next | 2 | 2 | `c` | 2 | 1 |
| next | 2 | 1 | `c` | 3 | 0 |
| hasNext | 3 | 0 | `False` | 3 | 0 |
| next | 3 | 0 | `" "` | 3 | 0 |

Ba dòng cuối cần chú ý: khi `idx` vượt cuối (`idx = 3 = len(pairs)`), ta **không** gán `remaining` mới, nên `remaining` giữ giá trị `0` (vừa giảm về 0). Từ đó `hasNext()` là `False` và `next()` trả dấu cách. Chạy trace để đối chiếu với bảng.

In [8]:
def trace_pairs(compressed, calls):
    it = StringIteratorPairs(compressed)
    print("pairs =", it.pairs)
    for op in calls:
        before = (it.idx, it.remaining)
        res = getattr(it, op)()
        print(f"{op:<8} idx,remaining trước={before} -> trả {res!r:<6} | sau=({it.idx}, {it.remaining})")

trace_pairs("a2b1c3", ["next", "next", "next", "hasNext", "next", "next", "next", "hasNext", "next"])

pairs = [('a', 2), ('b', 1), ('c', 3)]
next     idx,remaining trước=(0, 2) -> trả 'a'    | sau=(0, 1)
next     idx,remaining trước=(0, 1) -> trả 'a'    | sau=(1, 1)
next     idx,remaining trước=(1, 1) -> trả 'b'    | sau=(2, 3)
hasNext  idx,remaining trước=(2, 3) -> trả True   | sau=(2, 3)
next     idx,remaining trước=(2, 3) -> trả 'c'    | sau=(2, 2)
next     idx,remaining trước=(2, 2) -> trả 'c'    | sau=(2, 1)
next     idx,remaining trước=(2, 1) -> trả 'c'    | sau=(3, 0)
hasNext  idx,remaining trước=(3, 0) -> trả False  | sau=(3, 0)
next     idx,remaining trước=(3, 0) -> trả ' '    | sau=(3, 0)


Output khớp bảng: sau ký tự `c` cuối cùng, `idx = 3` (vượt cuối) và `remaining = 0` → `hasNext()` là `False`, `next()` trả `' '`. Thói quen tốt: lập bảng tay trước, rồi để trace bằng code xác nhận – nếu lệch, một trong hai đang sai.

`{res!r}` trong f-string in theo dạng `repr` (có dấu nháy) để nhìn thấy được dấu cách `' '`.

## 6. Cách 3 — Phân tích lười (lazy) ngay trên chuỗi nén

Không cần cả list cặp: chỉ giữ **vị trí đọc** `self.i` trong chuỗi nén, **ký tự hiện tại** `self.ch` và **số lần còn lại** `self.count`. Khi `count` về 0, đọc cặp kế tiếp từ chuỗi. Bộ nhớ phụ `O(1)` (ngoài chuỗi đầu vào).

Hàm phụ `_load()` (dấu `_` đầu tên = "dùng nội bộ") đọc **một** cặp tại `self.i`.

In [9]:
class StringIteratorLazy:
    def __init__(self, compressedString: str):
        self.s = compressedString
        self.i = 0          # vị trí đọc trong chuỗi nén
        self.ch = " "
        self.count = 0      # số lần còn lại của self.ch

    def _load(self):
        """Nếu count = 0, đọc cặp kế tiếp (bỏ qua cặp có số 0 nếu có)."""
        while self.count == 0 and self.i < len(self.s):
            self.ch = self.s[self.i]
            self.i += 1
            num = 0
            while self.i < len(self.s) and self.s[self.i].isdigit():
                num = num * 10 + int(self.s[self.i])
                self.i += 1
            self.count = num

    def next(self) -> str:
        if not self.hasNext():
            return " "
        self.count -= 1
        return self.ch

    def hasNext(self) -> bool:
        self._load()
        return self.count > 0


print(run_ops(StringIteratorLazy, OPS, ARGS))

[None, 'L', 'e', 'e', 't', 'C', 'o', True, 'd', True]


In [10]:
def trace_lazy(compressed, calls):
    it = StringIteratorLazy(compressed)
    for op in calls:
        res = getattr(it, op)()
        print(f"{op:<8} -> {res!r:<6} | i={it.i:<2} ch={it.ch!r} count={it.count}")

trace_lazy("x3Y12", ["next", "next", "next", "hasNext", "next", "next"])

next     -> 'x'    | i=2  ch='x' count=2
next     -> 'x'    | i=2  ch='x' count=1
next     -> 'x'    | i=2  ch='x' count=0
hasNext  -> True   | i=5  ch='Y' count=12
next     -> 'Y'    | i=5  ch='Y' count=11
next     -> 'Y'    | i=5  ch='Y' count=10


Quan sát: chỉ khi `x` hết (`count = 0`) và có lời gọi tiếp theo, `_load` mới đọc `Y12` (hai chữ số → 12). Đó là "lười": việc gì cần mới làm.

## 7. Cách 4 — Tách cặp bằng biểu thức chính quy `re` (gọn khi phỏng vấn)

Module chuẩn `re`: `re.findall(r"([A-Za-z])(\d+)", s)` trả về list các tuple `(chữ cái, chuỗi số)`.
- `[A-Za-z]`: một chữ cái; `\d+`: một hoặc nhiều chữ số; `( )`: nhóm cần lấy ra.

Ta kết hợp với `deque` để "lấy cặp đầu" bằng `popleft` trong `O(1)`.

In [11]:
import re
from collections import deque

print(re.findall(r"([A-Za-z])(\d+)", "L1e2t1C10"))


class StringIteratorRegex:
    def __init__(self, compressedString: str):
        self.q = deque([ch, int(k)] for ch, k in re.findall(r"([A-Za-z])(\d+)", compressedString) if int(k) > 0)

    def next(self) -> str:
        if not self.q:
            return " "
        front = self.q[0]
        front[1] -= 1
        if front[1] == 0:
            self.q.popleft()
        return front[0]

    def hasNext(self) -> bool:
        return bool(self.q)


print(run_ops(StringIteratorRegex, OPS, ARGS))

[('L', '1'), ('e', '2'), ('t', '1'), ('C', '10')]
[None, 'L', 'e', 'e', 't', 'C', 'o', True, 'd', True]


## 8. Code nộp LeetCode

Bản chính: danh sách cặp + con trỏ (rõ ràng, dễ debug). Trên LeetCode tên class phải đúng là `StringIterator`.

In [12]:
class StringIterator:

    def __init__(self, compressedString: str):
        self.pairs = []
        s, i, n = compressedString, 0, len(compressedString)
        while i < n:
            ch = s[i]
            i += 1
            num = 0
            while i < n and s[i].isdigit():
                num = num * 10 + int(s[i])
                i += 1
            if num > 0:
                self.pairs.append((ch, num))
        self.idx = 0
        self.remaining = self.pairs[0][1] if self.pairs else 0

    def next(self) -> str:
        if self.remaining == 0:
            return " "
        ch = self.pairs[self.idx][0]
        self.remaining -= 1
        if self.remaining == 0:
            self.idx += 1
            if self.idx < len(self.pairs):
                self.remaining = self.pairs[self.idx][1]
        return ch

    def hasNext(self) -> bool:
        return self.remaining > 0


got = run_ops(StringIterator, OPS, ARGS)
print(got)
assert got == EXPECTED
print("StringIterator PASS ví dụ đề")

[None, 'L', 'e', 'e', 't', 'C', 'o', True, 'd', True]
StringIterator PASS ví dụ đề


## 9. Vì sao đúng? (invariant)

Gọi `T` là chuỗi gốc (không bao giờ dựng ra). Với cách 2:

**Invariant:** phần **chưa đọc** của `T` luôn bằng
`pairs[idx][0] * remaining + pairs[idx+1][0] * pairs[idx+1][1] + ...`
và `remaining > 0` mỗi khi `idx < len(pairs)`.

- **Khởi tạo:** `idx = 0`, `remaining = pairs[0][1]` → phần chưa đọc = cả `T`. ✔
- **`next()`:** trả ký tự đầu của phần chưa đọc (`pairs[idx][0]`), giảm `remaining`. Nếu về 0, chuyển sang cặp kế với số lần đầy đủ → invariant giữ nguyên.
- **`hasNext()`:** phần chưa đọc rỗng ⇔ `idx == len(pairs)` ⇔ `remaining == 0` (nhờ vế thứ hai của invariant). ✔

Lọc bỏ cặp có số lần `0` (nếu dữ liệu lạ) để vế "`remaining > 0` khi còn cặp" luôn đúng.

## 10. Time / Space Complexity và so sánh

`L` = độ dài chuỗi nén (≤ 1000), `N` = độ dài chuỗi gốc (tổng các số lần, có thể ~`10^9 · 500`), `P` = số cặp (≤ `L/2`).

| Cách | `__init__` | `next` / `hasNext` | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Giải nén | `O(N)` | `O(1)` | `O(N)` | Hỏng khi số lần lớn |
| 2. Cặp + con trỏ | `O(L)` | `O(1)` | `O(P)` | Bản chính |
| 3. Lazy | `O(1)` | `O(1)` khấu hao (tổng đọc chuỗi `O(L)`) | `O(1)` phụ | Ít bộ nhớ nhất |
| 4. Regex + deque | `O(L)` | `O(1)` | `O(P)` | Gọn, cần biết regex |

"Khấu hao" (amortized): một lần `hasNext` có thể đọc nhiều ký tự, nhưng **tổng cộng** cả đời đối tượng chỉ đọc mỗi ký tự của chuỗi nén một lần.

## 11. Bộ test: mẫu đề, biên, đối chiếu giữa các cách

In [13]:
import random

classes = [StringIteratorExpand, StringIteratorPairs, StringIteratorLazy, StringIteratorRegex, StringIterator]

for cls in classes:
    assert run_ops(cls, OPS, ARGS) == EXPECTED, cls.__name__
print("Ví dụ đề: 5 cách PASS")

def drain(cls, s, extra=3):
    """Gọi next tới khi hết, rồi gọi thêm vài lần; trả về list kết quả và hasNext cuối."""
    it = cls(s)
    out = []
    while it.hasNext():
        out.append(it.next())
    out += [it.next() for _ in range(extra)]
    return "".join(out), it.hasNext()

edge = [
    ("a1", ("a   ", False)),
    ("a12", ("a" * 12 + "   ", False)),
    ("Z3z2", ("ZZZzz   ", False)),
    ("b10c1", ("b" * 10 + "c   ", False)),
]
for s, expected in edge:
    for cls in classes:
        assert drain(cls, s) == expected, (cls.__name__, s, drain(cls, s))
    print(f"{s!r:<8} PASS -> {expected}")

Ví dụ đề: 5 cách PASS
'a1'     PASS -> ('a   ', False)
'a12'    PASS -> ('aaaaaaaaaaaa   ', False)
'Z3z2'   PASS -> ('ZZZzz   ', False)
'b10c1'  PASS -> ('bbbbbbbbbbc   ', False)


In [14]:
rng = random.Random(604)
letters = "abcXYZ"
for _ in range(300):
    s = "".join(rng.choice(letters) + str(rng.randint(1, 15)) for _ in range(rng.randint(1, 5)))
    calls = [rng.choice(["next", "hasNext"]) for _ in range(rng.randint(1, 40))]
    results = []
    for cls in classes:
        it = cls(s)
        results.append([getattr(it, op)() for op in calls])
    assert all(r == results[0] for r in results), (s, calls)
print("300 kịch bản ngẫu nhiên: 5 cách cho cùng chuỗi kết quả — PASS")

300 kịch bản ngẫu nhiên: 5 cách cho cùng chuỗi kết quả — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ đọc **một** chữ số
`num = int(s[i])` → `"a12"` bị hiểu là `a` lặp 1 lần rồi gặp `"2"` như một "ký tự". Phải đọc **hết** chữ số liên tiếp.

In [15]:
def parse_one_digit_wrong(s):
    pairs = []
    for i in range(0, len(s) - 1, 2):
        pairs.append((s[i], int(s[i + 1])))
    return pairs

print("Sai với 'a12' :", parse_one_digit_wrong("a12"), "<- mất chữ số 2, a chỉ còn 1 lần")
print("Đúng với 'a12':", parse_pairs("a12"))
try:
    parse_one_digit_wrong("a12b3")
except ValueError as e:
    print("Sai với 'a12b3': ValueError ->", e)
print("Đúng với 'a12b3':", parse_pairs("a12b3"))

Sai với 'a12' : [('a', 1)] <- mất chữ số 2, a chỉ còn 1 lần
Đúng với 'a12': [('a', 12)]
Sai với 'a12b3': ValueError -> invalid literal for int() with base 10: 'b'
Đúng với 'a12b3': [('a', 12), ('b', 3)]


### Lỗi 2: Giải nén toàn bộ trong `__init__`
Chạy được ví dụ nhỏ nhưng **Memory Limit Exceeded** / quá thời gian khi số lần ~`10^9`.

### Lỗi 3: Trả về `""` (chuỗi rỗng) hoặc `None` khi hết
Đề yêu cầu **dấu cách** `" "`.

### Lỗi 4: `hasNext()` kiểm tra `idx < len(pairs)` nhưng quên chuyển cặp khi `remaining` về 0
Khi đó sau ký tự cuối, `idx` vẫn trỏ cặp cuối với `remaining = 0` → `hasNext()` sai thành `True`.

In [16]:
class WrongHasNext:
    def __init__(self, s):
        self.pairs = parse_pairs(s)
        self.idx, self.remaining = 0, self.pairs[0][1]

    def next(self):
        if self.remaining == 0 and self.idx + 1 < len(self.pairs):   # chuyển cặp "muộn"
            self.idx += 1
            self.remaining = self.pairs[self.idx][1]
        if self.remaining == 0:
            return " "
        self.remaining -= 1
        return self.pairs[self.idx][0]

    def hasNext(self):
        return self.idx < len(self.pairs)          # SAI

w = WrongHasNext("a1")
print("next:", repr(w.next()), "| hasNext (sai):", w.hasNext(), "| next:", repr(w.next()))

next: 'a' | hasNext (sai): True | next: ' '


### Lỗi 5: Đặt tên phương thức khác đề
LeetCode gọi đúng `next` và `hasNext` (chữ `N` hoa). Viết `has_next` sẽ bị lỗi `AttributeError` khi chấm.

## 13. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu bài Iterator/Design:** có `next()`, `hasNext()`; dữ liệu "nén" hoặc rất lớn; số lời gọi ít. → **Không vật chất hóa** (materialize) dữ liệu; giữ **con trỏ + bộ đếm** và tính dần.

**Liên hệ:**
- 900. RLE Iterator – `next(n)` bỏ qua `n` phần tử cùng lúc (trừ dồn vào bộ đếm).
- 281. Zigzag Iterator, 251. Flatten 2D Vector, 341. Flatten Nested List Iterator – đều là "con trỏ + trạng thái".
- 443. String Compression – chiều ngược lại: **nén** chuỗi.
- Python có sẵn khái niệm iterator: `iter()`, `next()`, và **generator** (`yield`). Ta có thể viết bản generator rất gọn.

In [17]:
def decompress_gen(compressed):
    """Generator: sinh từng ký tự, không bao giờ dựng chuỗi gốc."""
    for ch, k in parse_pairs(compressed):
        for _ in range(k):
            yield ch

g = decompress_gen("L1e2t1")
print(next(g), next(g), next(g), next(g))
print("Hết thì next(g, ' ') trả giá trị mặc định:", repr(next(g, " ")))

L e e t
Hết thì next(g, ' ') trả giá trị mặc định: ' '


## 14. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Đoán output.** Với `it = StringIterator("a2b1")`, chuỗi kết quả của các lời gọi `next, next, hasNext, next, hasNext, next` là gì?

**Bài 2 — Tính tay.** Viết `parse_pairs("Q10w1E205")`.

**Bài 3 — Hàm nén ngược:** `compress(text)` biến `"aaabcc"` → `"a3b1c2"`.

**Bài 4 — `skip(k)`:** thêm phương thức bỏ qua `k` ký tự (tối đa đến hết) trong `O(số cặp đi qua)`, không lặp `k` lần.

**Bài 5 — Bản generator đầy đủ:** class `StringIteratorGen` dùng generator bên trong; cần "nhìn trước" một ký tự để trả lời `hasNext`.

**Bài 6 — Mini challenge (RLE Iterator, LC 900):** cho mảng `encoding = [c0, v0, c1, v1, ...]` (số lần `c_i` của giá trị `v_i`); `next(n)` bỏ `n` phần tử và trả về phần tử cuối bị bỏ, hoặc `-1` nếu không đủ.

In [18]:
answer_ex1 = None    # TODO: list kết quả
answer_ex2 = None    # TODO: list cặp


def compress(text):
    # TODO: duyệt, đếm các ký tự giống nhau liên tiếp
    pass


class StringIteratorSkip(StringIteratorPairs):
    # Kế thừa (inheritance): dùng lại __init__, next, hasNext của StringIteratorPairs
    def skip(self, k):
        # TODO: trừ dần k vào remaining, chuyển cặp khi cần
        pass

# Kiểm tra sau khi làm:
# assert compress("aaabcc") == "a3b1c2"
# it = StringIteratorSkip("a3b2c1"); it.skip(4); assert it.next() == "b"

In [19]:
class StringIteratorGen:
    def __init__(self, compressedString):
        # TODO: tạo generator, đọc trước 1 ký tự vào self.peek (hoặc None)
        pass

    def next(self):
        # TODO
        pass

    def hasNext(self):
        # TODO
        pass


class RLEIterator:
    def __init__(self, encoding):
        # TODO: lưu encoding và con trỏ
        pass

    def next(self, n):
        # TODO
        pass

# Kiểm tra sau khi làm:
# assert run_ops(StringIteratorGen, OPS, ARGS) == EXPECTED
# r = RLEIterator([3, 8, 0, 9, 2, 5]); assert [r.next(2), r.next(1), r.next(1), r.next(2)] == [8, 8, 5, -1]

## 15. Gợi ý 3 mức

**Bài 1:** (1) Chuỗi gốc `aab`. (2) Sau hai `next` còn `b`. (3) Sau `b` thì `hasNext` là `False`, `next` trả `' '`.

**Bài 2:** (1) Đọc hết chữ số sau mỗi chữ cái. (2) `205` là một số. (3) Ba cặp.

**Bài 3:** (1) Đếm "chuỗi chạy" (run) các ký tự giống nhau liên tiếp. (2) Khi ký tự đổi thì ghi `ch + str(count)`. (3) Nhớ ghi run cuối cùng sau vòng lặp.

**Bài 4:** (1) Trong mỗi bước: nếu `k < remaining` thì chỉ trừ; ngược lại trừ `remaining` khỏi `k` và chuyển cặp. (2) Dừng khi `k == 0` hoặc hết cặp. (3) Khi chuyển cặp giữ đúng quy ước `remaining = 0` khi đã hết.

**Bài 5:** (1) Generator không có `hasNext`, nên đọc trước một phần tử. (2) `next(gen, None)` trả `None` khi hết. (3) `next()` trả `self.peek` rồi đọc phần tử kế vào `self.peek`.

**Bài 6:** (1) Con trỏ `i` vào cặp `(encoding[i], encoding[i+1])`. (2) Nếu `encoding[i] >= n` thì trừ và trả `encoding[i+1]`; ngược lại `n -= encoding[i]`, `i += 2`. (3) Hết mảng → `-1`.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [20]:
it = StringIterator("a2b1")
ans_ex1 = [it.next(), it.next(), it.hasNext(), it.next(), it.hasNext(), it.next()]   # ['a','a',True,'b',False,' ']
ans_ex2 = parse_pairs("Q10w1E205")                                                    # [('Q',10),('w',1),('E',205)]


def ans_compress(text):
    if not text:
        return ""
    parts = []
    run_ch, run_len = text[0], 1
    for ch in text[1:]:
        if ch == run_ch:
            run_len += 1
        else:
            parts.append(run_ch + str(run_len))
            run_ch, run_len = ch, 1
    parts.append(run_ch + str(run_len))
    return "".join(parts)


class AnsStringIteratorSkip(StringIteratorPairs):
    def skip(self, k):
        while k > 0 and self.remaining > 0:
            if k < self.remaining:
                self.remaining -= k
                k = 0
            else:
                k -= self.remaining
                self.remaining = 0
                self.idx += 1
                if self.idx < len(self.pairs):
                    self.remaining = self.pairs[self.idx][1]


class AnsStringIteratorGen:
    def __init__(self, compressedString):
        self.gen = decompress_gen(compressedString)
        self.peek = next(self.gen, None)

    def next(self):
        if self.peek is None:
            return " "
        ch = self.peek
        self.peek = next(self.gen, None)
        return ch

    def hasNext(self):
        return self.peek is not None


class AnsRLEIterator:
    def __init__(self, encoding):
        self.enc = encoding
        self.i = 0

    def next(self, n):
        while self.i < len(self.enc):
            if self.enc[self.i] >= n:
                self.enc[self.i] -= n
                return self.enc[self.i + 1]
            n -= self.enc[self.i]
            self.i += 2
        return -1


print("Bài 1:", ans_ex1)
print("Bài 2:", ans_ex2)
assert ans_ex1 == ["a", "a", True, "b", False, " "]
assert ans_ex2 == [("Q", 10), ("w", 1), ("E", 205)]
assert ans_compress("aaabcc") == "a3b1c2" and ans_compress("LeetCode") == "L1e2t1C1o1d1e1"
assert drain(StringIterator, ans_compress("zzzzzzzzzzzzq"))[0].strip() == "zzzzzzzzzzzzq"
sk = AnsStringIteratorSkip("a3b2c1"); sk.skip(4); assert sk.next() == "b" and sk.next() == "c"
sk = AnsStringIteratorSkip("a3"); sk.skip(100); assert sk.hasNext() is False and sk.next() == " "
assert run_ops(AnsStringIteratorGen, OPS, ARGS) == EXPECTED
r = AnsRLEIterator([3, 8, 0, 9, 2, 5])
assert [r.next(2), r.next(1), r.next(1), r.next(2)] == [8, 8, 5, -1]
print("Đáp án: tất cả test PASS")

Bài 1: ['a', 'a', True, 'b', False, ' ']
Bài 2: [('Q', 10), ('w', 1), ('E', 205)]
Đáp án: tất cả test PASS


## 17. Template nhớ nhanh và checklist submit

```python
class StringIterator:
    def __init__(self, compressedString: str):
        self.pairs, s, i = [], compressedString, 0
        while i < len(s):
            ch, i, num = s[i], i + 1, 0
            while i < len(s) and s[i].isdigit():
                num, i = num * 10 + int(s[i]), i + 1
            if num:
                self.pairs.append((ch, num))
        self.idx = 0
        self.remaining = self.pairs[0][1] if self.pairs else 0

    def next(self) -> str:
        if self.remaining == 0:
            return " "
        ch = self.pairs[self.idx][0]
        self.remaining -= 1
        if self.remaining == 0:                    # hết cặp -> sang cặp sau ngay
            self.idx += 1
            if self.idx < len(self.pairs):
                self.remaining = self.pairs[self.idx][1]
        return ch

    def hasNext(self) -> bool:
        return self.remaining > 0
```

Câu nhớ: **"đừng giải nén – giữ (cặp hiện tại, còn bao nhiêu)"**.

### Checklist
- [ ] Đọc số **nhiều chữ số** (`num = num * 10 + digit`)?
- [ ] Không dựng chuỗi gốc (số lần tới `10^9`)?
- [ ] Hết ký tự thì `next()` trả `" "` (dấu cách)?
- [ ] Chuyển cặp **ngay** khi `remaining` về 0 để `hasNext` đơn giản và đúng?
- [ ] Tên class/method đúng: `StringIterator`, `next`, `hasNext`?